### Data Ingestion 

In [ ]:
#Create a Document Manually
from langchain_core.documents import Document

doc = Document(
    page_content="Python is a programming language",
    metadata={"source": "manual", "page": 1}
)

print(doc)

page_content='Python is a programming language' metadata={'source': 'manual', 'page': 1}


In [19]:
#Load single file 
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/data.txt")
documents = loader.load()

print(documents)

[Document(metadata={'source': 'data/data.txt'}, page_content='RAG stands for Retrieval Augmented Generation.\nIt retrieves relevant information and uses LLM to generate answers.\nFAISS is a vector database used for similarity search.\nLangChain helps orchestrate the RAG pipeline.')]


In [ ]:
#Load Multiple Files
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data",
    glob="**/*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(len(documents))

1


In [ ]:
#Load PDF Files
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("data/PROJECT_II_FINAL_Santhosh.docx.pdf")
documents = loader.load()

print(documents[0].page_content)

A project report on 
 
A REAL-TIME ML-BASED ADAPTIVE 
SECURITY SERVICE FOR BANKING 
APPLICATIONS 
Submitted in partial fulfillment for the award of the degree of 
 
Bachelor of Technology in Computer Science 
Engineering with specialization in 
Artificial Intelligence and Machine 
Learning 
by 
 
SANTHOSH KUMAR S (22BAI1040) 
 
 
 
 
SCHOOL OF COMPUTER SCIENCE AND 
ENGINEERING


### Embedding And Vector Store DB


In [10]:
import numpy as np
import os
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the embedding model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dim: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for list of texts"""

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)

        print(f"Generated embeddings shape: {embeddings.shape}")
        return embeddings
    
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10211.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dim: 384


C:\Users\SANTHOSH\AppData\Local\Temp\ipykernel_13176\4085001061.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dim: {self.model.get_sentence_embedding_dimension()}")


### Vectore Store 

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized: {self.collection_name}")
            print(f"Existing docs: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
        

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and embeddings to vector DB"""

        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings must match")

        print(f"Adding {len(documents)} documents...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Text
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to ChromaDB
        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=documents_text
        )

        print(f"Added {len(documents)} documents successfully")
        print(f"Total docs: {self.collection.count()}")


vectorstore=VectorStore()
vectorstore


Vector store initialized: pdf_documents
Existing docs: 0


### RAG Retriever 

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for a query"""

        print(f"Query: {query}")

        # Convert query → embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in ChromaDB
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        # Format results
        retrieved_docs = []

        for i in range(len(results["documents"][0])):
            doc = {
                "content": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "score": results["distances"][0][i]
            }

            # Optional filtering
            if doc["score"] >= score_threshold:
                retrieved_docs.append(doc)

        print(f"Retrieved {len(retrieved_docs)} documents")

        return retrieved_docs
    
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve("what is attention")

Query: what is attention
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.41it/s]

Generated embeddings shape: (1, 384)
Retrieved 0 documents


[]

### Full Code With Chunking 

In [ ]:
#Create a Document Manually
import os
import numpy as np
import chromadb
import uuid

from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from typing import List, Dict, Any

doc = Document(
    page_content="Python is a programming language",
    metadata={"source": "manual", "page": 1}
)

print(doc) 

#Load single file 
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/data.txt")
documents = loader.load()

#Load Multiple Files
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data",
    glob="**/*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(len(documents))

#Load PDF Files
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(r"C:\College\RAG\notebook\data\Paper 2 (2).pdf")
documents = loader.load()

print(documents[0].page_content)

print(documents)



from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunked_documents = splitter.split_documents(documents)

print(f"Total chunks: {len(chunked_documents)}")


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the embedding model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dim: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for list of texts"""

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)

        print(f"Generated embeddings shape: {embeddings.shape}")
        return embeddings
    
embedding_manager=EmbeddingManager()
embedding_manager



texts = [doc.page_content for doc in chunked_documents]


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized: {self.collection_name}")
            print(f"Existing docs: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
        

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and embeddings to vector DB"""

        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings must match")

        print(f"Adding {len(documents)} documents...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Text
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to ChromaDB
        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=documents_text
        )

        print(f"Added {len(documents)} documents successfully")
        print(f"Total docs: {self.collection.count()}")


vectorstore=VectorStore()
vectorstore



embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunked_documents, embeddings)


class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for a query"""

        print(f"Query: {query}")

        # Convert query → embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #  Search in ChromaDB
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        # Format results
        retrieved_docs = []

        for i in range(len(results["documents"][0])):
            doc = {
                "content": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "score": results["distances"][0][i]
            }

            if doc["score"] >= score_threshold:
                retrieved_docs.append(doc)

        print(f"Retrieved {len(retrieved_docs)} documents")

        return retrieved_docs
    
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

page_content='Python is a programming language' metadata={'source': 'manual', 'page': 1}
1
https://iaeme.com/Home/journal/IJCET 
1051 
editor@iaeme.com 
International Journal of Computer Engineering and Technology (IJCET)  
Volume 16, Issue 1, Jan-Feb 2025, pp. 1051-1064, Article ID: IJCET_16_01_082 
Available online at https://iaeme.com/Home/issue/IJCET?Volume=16&Issue=1 
ISSN Print: 0976-6367; ISSN Online: 0976-6375; Journal ID: 5751-5249 
Impact Factor (2025): 18.59 (Based on Google Scholar Citation) 
DOI: https://doi.org/10.34218/IJCET_16_01_082 
 
 
© IAEME Publication 
REAL-TIME FRAUD DETECTION IN BANKING 
WITH GENERATIVE ARTIFICIAL 
INTELLIGENCE 
Venkata Rupesh Kumar Dabbir 
USA. 
 
ABSTRACT 
The increase in digital transactions, cases of banking fraud have also increased. In 
this scenario, real-time fraud detection methods are essential for financial institutions. 
Outdated techniques can struggle to keep pace with these continually evolving fraud 
tactics. One potential solut

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6657.83it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\SANTHOSH\AppData\Local\Temp\ipykernel_27048\2935080679.py:75: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dim: {self.model.get_sentence_embedding_dimension()}")


Model loaded successfully. Embedding dim: 384
Vector store initialized: pdf_documents
Existing docs: 76
Generating embeddings for 14 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.27it/s]

Generated embeddings shape: (14, 384)
Adding 14 documents...
Added 14 documents successfully
Total docs: 90


### Integration Vector DB Context Pipeline With LLM O/P

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI   

load_dotenv()

client = OpenAI(api_key=os.getenv(""))

def rag_qa(query: str, retriever, top_k: int = 3):
    #  Retrieve context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found."

    # Prompt
    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    # OpenAI LLM call
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [9]:
answer = rag_qa("What is fraud detection?", rag_retriever)

print(answer)

Query: What is fraud detection?
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.25it/s]

Generated embeddings shape: (1, 384)
Retrieved 3 documents


Fraud detection refers to the process of identifying and preventing fraudulent activities, particularly in financial transactions. It involves using advanced technologies, such as machine learning and artificial intelligence, to analyze transaction data in real-time for patterns or anomalies that may indicate fraud. The goal is to flag potentially fraudulent transactions before they result in financial losses, while balancing the need to minimize false positives and negatives.


### Enhanced RAG

In [ ]:
def rag_advanced(query, retriever, top_k=5, min_score=0.2, return_context=False):
    """
    Enhanced RAG:
    - Retrieval + filtering
    - OpenAI generation
    - Sources + confidence
    """

    print(f"\n Query: {query}")

    # Retrieve relevant chunks
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    #  Handle no results
    if not results:
        return {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }

    # Build context
    context = "\n\n".join([doc["content"] for doc in results])

    # Build sources
    sources = []
    for doc in results:
        sources.append({
            "source": doc["metadata"].get("source", "unknown"),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["score"],
            "preview": doc["content"][:120] + "..."
        })

    #  Confidence score
    confidence = max([doc["score"] for doc in results])

    # Prompt
    prompt = f"""
Use ONLY the context below to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

    #  OpenAI call (NO llm variable needed)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer only from the provided context."},
            {"role": "user", "content": prompt}
        ]
    )

    answer = response.choices[0].message.content

    #  Output
    output = {
        "answer": answer,
        "sources": sources,
        "confidence": confidence
    }

    #  Optional context
    if return_context:
        output["context"] = context

    return output

In [14]:
result = rag_advanced(
    "How does AI help in fraud detection?",
    rag_retriever,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("\nAnswer:\n", result["answer"])
print("\nSources:\n", result["sources"])
print("\nConfidence:\n", result["confidence"])
print("\nContext Preview:\n", result["context"][:300])


🔍 Query: How does AI help in fraud detection?
Query: How does AI help in fraud detection?
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.46it/s]

Generated embeddings shape: (1, 384)
Retrieved 3 documents



Answer:
 AI helps in fraud detection by utilizing generative artificial intelligence to analyze vast amounts of data in real-time, effectively identifying patterns and anomalies that may indicate fraudulent activities. It enhances the accuracy of detection, reducing the false positive rate, and allows for preemptive measures against evolving fraud methodologies. Additionally, advanced AI models, such as deep learning, automate complex tasks and improve decision-making, further increasing the efficiency of identifying fraud in the banking sector.

Sources:
 [{'source': 'C:\\College\\RAG\\notebook\\data\\Paper 2 (2).pdf', 'page': 9, 'score': 0.6115931272506714, 'preview': 'Venkata Rupesh Kumar Dabbir \nhttps://iaeme.com/Home/journal/IJCET \n1060 \neditor@iaeme.com \n4. Result and Discussion \n4.1...'}, {'source': 'C:\\College\\RAG\\notebook\\data\\Paper 2 (2).pdf', 'page': 11, 'score': 0.6496809720993042, 'preview': 'Venkata Rupesh Kumar Dabbir \nhttps://iaeme.com/Home/journal/IJCET \n1